# Faruq mask-geometry audit (train/validation only)

Notebook ini mengaudit apakah polygon mask Faruq masih sejajar dengan gambar setelah orientasi EXIF/COCO. Ia **tidak melatih model, tidak menjalankan inference, tidak memperbaiki anotasi, dan tidak membaca split test**. Dataset Roboflow dapat terunduh lengkap, tetapi kode audit hanya membuka `train` dan `valid`.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone_command = [
    'git', 'clone', '--depth', '1', '--branch', BRANCH,
    'https://github.com/ediprin/coffee-bean-detection.git', str(REPO),
]
for attempt in range(1, 4):
    result = subprocess.run(clone_command)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), 'roboflow'], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
import coffee_detector
print('IMPORT:', coffee_detector.__file__)


In [ ]:
RAW_ROOT = Path('/content/faruq-segmentation-raw')
OUTPUT_ROOT = Path('/content/drive/MyDrive/Coffee_Bean_Detection/evidence/faruq-mask-geometry-audit-v1')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def has_coco_development(root):
    return any(
        any(path.is_file() for path in (root / split).glob('*.json'))
        for split in ('train', 'valid', 'val')
    )

if not has_coco_development(RAW_ROOT):
    from roboflow import Roboflow
    api_key = userdata.get('roboflow')
    assert api_key, "Tambahkan Colab secret bernama 'roboflow' dan aktifkan akses notebook."
    rf = Roboflow(api_key=api_key)
    version = rf.workspace('situju-kamkape').project('robusta_sni_dataset-hr9ci').version(1)
    downloaded = version.download('coco-segmentation', location=str(RAW_ROOT))
    RAW_ROOT = Path(downloaded.location)

assert has_coco_development(RAW_ROOT), f'COCO train/valid tidak ditemukan: {RAW_ROOT}'
print('RAW ROOT   :', RAW_ROOT)
print('OUTPUT ROOT:', OUTPUT_ROOT)
print('Catatan: folder test mungkin ikut terunduh, tetapi tidak akan dibaca oleh audit.')


In [ ]:
import json
from coffee_detector.audit_faruq_mask_geometry import audit_faruq_mask_geometry

summary = audit_faruq_mask_geometry(
    RAW_ROOT,
    OUTPUT_ROOT,
    score_long_side=192,
    min_improvement=0.02,
    contact_sheet_limit=24,
)
assert summary['training_executed'] is False
assert summary['inference_executed'] is False
assert summary['test_images_accessed'] is False
assert summary['splits_accessed'] == ['train', 'val']
print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
from IPython.display import display
from PIL import Image

summary_path = OUTPUT_ROOT / 'faruq_geometry_audit_summary.json'
assert summary_path.is_file(), summary_path
sheet_path = OUTPUT_ROOT / 'flagged_orientation_contact_sheet.jpg'
if sheet_path.is_file():
    display(Image.open(sheet_path))
else:
    print('Tidak ada kasus yang melewati ambang flag; contact sheet tidak dibuat.')
print('SUMMARY:', summary_path)
print('Kirim summary JSON dan contact sheet. Jangan repair atau training sebelum overlay dinilai.')
